# nb10 -- Pathway / complex / regulatory validation
Checks whether the model's non-cognate top-predictor genes are known biological partners of the cognate gene, using three complementary sources:

1. **STRING** -- functional/physical association, broad coverage. This time we pull the per-evidence-channel scores (experiments, curated databases, co-expression, text-mining, etc.), not just the combined score, so a hit backed by hard experimental evidence can be told apart from one that's mostly literature text-mining.
2. **OmniPath complexes** -- widened to `CORUM, ComplexPortal, hu.MAP` (CORUM alone, tried previously, returned 0/60 -- its curation scope is narrower than the query set needed).
3. **TRRUST** -- literature-curated transcription-factor -> target-gene regulatory interactions. Added specifically because the FOXP3 -> IL2RA hit from the STRING pass is a regulatory relationship, not a physical complex, and neither STRING's association score nor a complex database is really built to confirm that kind of relationship on its own.

Scope: the 60 proteins that are neither in the trustworthy core (already validated, nb09) nor flagged as artifacts (already excluded, nb09) -- i.e. `cognate_rank > 1` and not artifact-flagged.

## Environment setup

In [2]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/covid_citeseq_project')
else:
    BASE_PATH = Path('..')

print(f"Running on {'Colab' if IN_COLAB else 'local'} | BASE_PATH = {BASE_PATH}")

Mounted at /content/drive
Running on Colab | BASE_PATH = /content/drive/MyDrive/covid_citeseq_project


## Imports and config

In [3]:
import time
import io

import numpy as np
import pandas as pd
import requests

NB09_RESULTS_DIR = BASE_PATH / 'results' / 'sparse_mlp_comparison'
RESULTS_DIR       = BASE_PATH / 'results' / 'pathway_validation'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

STRING_API_BASE  = 'https://string-db.org/api'
STRING_SPECIES   = 9606  # human
STRING_MIN_SCORE = 0.4   # STRING's own 'medium confidence' cutoff, 0-1 scale
REQUEST_DELAY_SEC = 1.0  # be polite to the STRING API

OMNIPATH_COMPLEX_RESOURCES = 'CORUM,ComplexPortal,hu.MAP'  # widened from CORUM alone

TRRUST_URL = 'https://www.grnpedia.org/trrust/data/trrust_rawdata.human.tsv'

## Load nb09 outputs and build the query set
Same scoping logic as before: everything that's neither trustworthy-core nor artifact-flagged.

In [4]:
trustworthy_core = pd.read_csv(NB09_RESULTS_DIR / 'trustworthy_core_cognate_pairs.csv')
artifact_flags    = pd.read_csv(NB09_RESULTS_DIR / 'artifact_flags.csv')

artifact_proteins = set(artifact_flags.loc[artifact_flags['likely_artifact'], 'protein'])
trustworthy_proteins = set(trustworthy_core.loc[trustworthy_core['trustworthy'], 'protein'])

query_set = artifact_flags[
    (~artifact_flags['protein'].isin(artifact_proteins))
    & (~artifact_flags['protein'].isin(trustworthy_proteins))
    & (artifact_flags['cognate_rank'] > 1)
].copy()

print(f'Trustworthy core (skipped): {len(trustworthy_proteins)}')
print(f'Artifact-flagged (excluded): {len(artifact_proteins)}')
print(f'Query set: {len(query_set)}')

Trustworthy core (skipped): 34
Artifact-flagged (excluded): 67
Query set: 60


## 1. STRING check -- combined score AND per-evidence-channel breakdown
STRING's `score` is a combined score across several independent evidence channels. `escore` (experimental) and `dscore` (curated databases) are hard-evidence channels; `tscore` (text-mining) is the weakest, most inflatable one. A pair whose combined score is driven mostly by `tscore` with near-zero `escore`/`dscore` is a much softer claim than one with real experimental or database backing.

In [5]:
def get_string_partners(gene_symbol: str, species: int = STRING_SPECIES, limit: int = 50) -> pd.DataFrame:
    """Fetch a gene's top STRING interaction partners, including per-channel evidence scores."""
    url = f'{STRING_API_BASE}/tsv/interaction_partners'
    params = {'identifiers': gene_symbol, 'species': species, 'limit': limit}
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        if not resp.text.strip():
            return pd.DataFrame()
        return pd.read_csv(io.StringIO(resp.text), sep='\t')
    except (requests.RequestException, pd.errors.ParserError):
        return pd.DataFrame()


unique_query_genes = pd.unique(query_set[['cognate_gene', 'top_predictor_gene']].values.ravel())
print(f'Fetching STRING partners for {len(unique_query_genes)} unique genes...')

string_partner_cache = {}
for i, gene in enumerate(unique_query_genes):
    string_partner_cache[gene] = get_string_partners(gene)
    time.sleep(REQUEST_DELAY_SEC)
    if (i + 1) % 10 == 0:
        print(f'  {i + 1}/{len(unique_query_genes)}...')

print('STRING fetch complete.')
if string_partner_cache:
    sample = next((df for df in string_partner_cache.values() if not df.empty), pd.DataFrame())
    print('Columns returned:', sample.columns.tolist())

Fetching STRING partners for 84 unique genes...
  10/84...
  20/84...
  30/84...
  40/84...
  50/84...
  60/84...
  70/84...
  80/84...
STRING fetch complete.
Columns returned: ['stringId_A', 'stringId_B', 'preferredName_A', 'preferredName_B', 'ncbiTaxonId', 'score', 'nscore', 'fscore', 'pscore', 'ascore', 'escore', 'dscore', 'tscore']


In [6]:
EVIDENCE_COLS = ['score', 'escore', 'dscore', 'tscore', 'ascore', 'nscore', 'fscore', 'pscore']


def check_string_interaction(gene_a: str, gene_b: str) -> dict:
    """Check whether gene_b appears among gene_a's cached STRING partners; return all evidence scores."""
    partners = string_partner_cache.get(gene_a, pd.DataFrame())
    if not partners.empty and 'preferredName_B' in partners.columns:
        match = partners[partners['preferredName_B'].str.upper() == gene_b.upper()]
        if not match.empty:
            return {col: float(match[col].iloc[0]) for col in EVIDENCE_COLS if col in match.columns}
    return {}


string_rows = []
for _, row in query_set.iterrows():
    evidence = check_string_interaction(row['cognate_gene'], row['top_predictor_gene'])
    if not evidence:
        evidence = check_string_interaction(row['top_predictor_gene'], row['cognate_gene'])
    result = {
        'protein': row['protein'], 'cognate_gene': row['cognate_gene'],
        'top_predictor_gene': row['top_predictor_gene'],
        'string_known_interaction': bool(evidence),
    }
    for col in EVIDENCE_COLS:
        result[f'string_{col}'] = evidence.get(col, np.nan)
    string_rows.append(result)

string_results = pd.DataFrame(string_rows)
string_results['string_high_confidence'] = string_results['string_score'] >= STRING_MIN_SCORE
string_results['string_hard_evidence'] = (
    (string_results['string_escore'].fillna(0) >= STRING_MIN_SCORE)
    | (string_results['string_dscore'].fillna(0) >= STRING_MIN_SCORE)
)
string_results['string_textmining_only'] = (
    string_results['string_known_interaction']
    & ~string_results['string_hard_evidence']
    & (string_results['string_tscore'].fillna(0) >= STRING_MIN_SCORE)
)

print(f"STRING-supported pairs: {string_results['string_known_interaction'].sum()} / {len(string_results)}")
print(f"  ...with hard evidence (experiments or curated database): {string_results['string_hard_evidence'].sum()}")
print(f"  ...text-mining evidence only: {string_results['string_textmining_only'].sum()}")
string_results[string_results['string_known_interaction']].sort_values('string_score', ascending=False)

STRING-supported pairs: 19 / 60
  ...with hard evidence (experiments or curated database): 8
  ...text-mining evidence only: 9


,protein,cognate_gene,top_predictor_gene,string_known_interaction,string_score,string_escore,string_dscore,string_tscore,string_ascore,string_nscore,string_fscore,string_pscore,string_high_confidence,string_hard_evidence,string_textmining_only
6,AB_CD8,CD8A,CD8B,True,0.999,0.691,0.9,0.892,0.819,0.0,0.0,0.000,True,True,False
56,AB_KLRD1,KLRD1,KLRC1,True,0.999,0.982,0.9,0.986,0.096,0.0,0.0,0.000,True,True,False
7,AB_CD19,CD19,CD79A,True,0.998,0.292,0.5,0.970,0.829,0.0,0.0,0.000,True,True,False
13,AB_CD4,CD4,CD8B,True,0.980,0.000,0.9,0.785,0.166,0.0,0.0,0.000,True,True,False
16,AB_CD25,IL2RA,FOXP3,True,0.970,0.288,0.5,0.916,0.142,0.0,0.0,0.000,True,True,False
49,AB_CD22,CD22,MS4A1,True,0.928,0.092,0.0,0.807,0.542,0.0,0.0,0.215,True,False,True
35,AB_ITGB7,ITGB7,ITGB1,True,0.913,0.000,0.9,0.139,0.000,0.0,0.0,0.075,True,True,False
20,AB_FCGR2A,FCGR2A,FCGR2B,True,0.892,0.471,0.0,0.706,0.356,0.0,0.0,0.053,True,True,False
37,AB_SELP,SELP,PPBP,True,0.887,0.000,0.0,0.862,0.219,0.0,0.0,0.000,True,False,True
32,AB_CD21,CR2,CD79A,True,0.886,0.000,0.0,0.829,0.363,0.0,0.0,0.000,True,False,True


## 2. OmniPath complexes, widened resources
CORUM alone (tried in the previous version of this notebook) returned 0/60 -- its curation scope didn't happen to overlap this query set. Widening to CORUM + Complex Portal + hu.MAP pulls in complexes from broader curation criteria and a different experimental method (hu.MAP is AP-MS-based, not literature curation).

In [11]:
def load_omnipath_complexes_all() -> pd.DataFrame:
    """Fetch all OmniPath complexes (no server-side resource filter, which is
    currently 502ing for some resources), then filter by source client-side.
    """
    local_path = RESULTS_DIR / 'omnipath_complexes_all.tsv'
    if local_path.exists():
        return pd.read_csv(local_path, sep='\t')
    url = 'https://omnipathdb.org/complexes'
    try:
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()
        df = pd.read_csv(io.StringIO(resp.text), sep='\t')
        df.to_csv(local_path, sep='\t', index=False)
        return df
    except (requests.RequestException, pd.errors.ParserError) as e:
        print(f'OmniPath fetch failed ({e}).')
        return pd.DataFrame()


all_complexes = load_omnipath_complexes_all()
print(f'Total complexes (all resources): {len(all_complexes)}')
print('sources column sample:', all_complexes['sources'].head(3).tolist() if not all_complexes.empty else None)

wanted = {'CORUM', 'ComplexPortal', 'hu.MAP', 'hu.MAP2', 'hu.MAP3'}
complexes = all_complexes[
    all_complexes['sources'].apply(lambda s: bool(wanted & set(str(s).split(';'))))
]
print(f'Filtered to {wanted}: {len(complexes)}')

Total complexes (all resources): 37629
sources column sample: ['hu.MAP;SIGNOR;CORUM;ComplexPortal;Compleat;hu.MAP2;SPIKE;PDB', 'SIGNOR', 'SIGNOR']
Filtered to {'CORUM', 'hu.MAP', 'hu.MAP3', 'ComplexPortal', 'hu.MAP2'}: 15462


In [13]:
def build_gene_to_complexes(complexes: pd.DataFrame, subunit_col: str = 'components_genesymbols',
                             name_col: str = 'name') -> dict:
    """Map each gene symbol to the set of complex names it's a subunit of. Genes are underscore-joined."""
    gene_to_complexes = {}
    if complexes.empty or subunit_col not in complexes.columns:
        return gene_to_complexes
    for _, row in complexes.iterrows():
        subunits = str(row[subunit_col]).split('_')
        complex_name = row.get(name_col)
        complex_name = 'unnamed' if pd.isna(complex_name) else str(complex_name)
        for gene in subunits:
            gene = gene.strip().upper()
            gene_to_complexes.setdefault(gene, set()).add(complex_name)
    return gene_to_complexes


def check_shared_complex(gene_a: str, gene_b: str) -> tuple[bool, str]:
    """Check whether two genes share at least one complex."""
    shared = gene_to_complexes.get(gene_a.upper(), set()) & gene_to_complexes.get(gene_b.upper(), set())
    return (len(shared) > 0, '; '.join(sorted(str(s) for s in shared)))


gene_to_complexes = build_gene_to_complexes(complexes)
print(f'Genes indexed: {len(gene_to_complexes)}')


complex_rows = []
for _, row in query_set.iterrows():
    shared, names = check_shared_complex(row['cognate_gene'], row['top_predictor_gene'])
    complex_rows.append({
        'protein': row['protein'], 'shared_complex': shared, 'complex_names': names,
    })

complex_results = pd.DataFrame(complex_rows)
print(f"Complex-supported pairs: {complex_results['shared_complex'].sum()} / {len(complex_results)}")
complex_results[complex_results['shared_complex']]

Genes indexed: 11719
Complex-supported pairs: 12 / 60


,protein,shared_complex,complex_names
2,AB_CD47,True,unnamed
6,AB_CD8,True,CD8_receptor
9,AB_TNFRSF17,True,unnamed
11,AB_CD7,True,unnamed
14,AB_CD44,True,unnamed
20,AB_FCGR2A,True,unnamed
23,AB_CD27,True,unnamed
24,AB_LAMP1,True,unnamed
48,AB_CD15,True,unnamed
54,AB_SLAMF7,True,unnamed


## 3. TRRUST -- transcription factor -> target regulatory check
Checks whether either gene in a pair is a known TF that regulates the other, in either direction (cognate gene regulating the model's pick, or vice versa -- e.g. FOXP3 -> IL2RA). If the primary URL is unreachable, download `trrust_rawdata.human.tsv` manually from https://www.grnpedia.org/trrust/ and place it at `RESULTS_DIR / 'trrust_rawdata.human.tsv'`.

In [14]:
TRRUST_MIRROR_URL = 'https://github.com/Searchlight2/Searchlight2/raw/refs/heads/master/upstream_regulator_databases/trrust.human.tsv'

def load_trrust() -> pd.DataFrame:
    """Load TRRUST human TF-target data. Tries the primary site, falls back to a
    GitHub mirror (grnpedia.org has a known outage), then a local file.
    """
    local_path = RESULTS_DIR / 'trrust_rawdata.human.tsv'
    if local_path.exists():
        return pd.read_csv(local_path, sep='\t', header=None, names=['tf', 'target', 'regtype', 'references'])

    for url in [TRRUST_URL, TRRUST_MIRROR_URL]:
        try:
            resp = requests.get(url, timeout=15)
            resp.raise_for_status()
            df = pd.read_csv(io.StringIO(resp.text), sep='\t', header=None, names=['tf', 'target', 'regtype', 'references'])
            df.to_csv(local_path, sep='\t', index=False, header=False)
            print(f'Loaded from {url}')
            return df
        except (requests.RequestException, pd.errors.ParserError) as e:
            print(f'{url} failed ({e})')

    print('Both sources failed -- download manually, see markdown above.')
    return pd.DataFrame()


trrust = load_trrust()
print(f'TRRUST interactions loaded: {len(trrust)}')
if not trrust.empty:
    print(trrust.head(3))

https://www.grnpedia.org/trrust/data/trrust_rawdata.human.tsv failed (HTTPSConnectionPool(host='www.grnpedia.org', port=443): Max retries exceeded with url: /trrust/data/trrust_rawdata.human.tsv (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7b0f5291a150>, 'Connection to www.grnpedia.org timed out. (connect timeout=15)')))
Loaded from https://github.com/Searchlight2/Searchlight2/raw/refs/heads/master/upstream_regulator_databases/trrust.human.tsv
TRRUST interactions loaded: 9398
     tf  target     regtype  references
0  AATF     BAX  Repression         NaN
1  AATF  CDKN1A     Unknown         NaN
2  AATF    KLK3     Unknown         NaN


In [15]:
def check_trrust_regulation(gene_a: str, gene_b: str) -> dict:
    """Check whether gene_a regulates gene_b, or gene_b regulates gene_a, per TRRUST."""
    if trrust.empty:
        return {'trrust_regulatory_pair': False, 'trrust_direction': None, 'trrust_regtype': None}
    a_to_b = trrust[(trrust['tf'].str.upper() == gene_a.upper()) & (trrust['target'].str.upper() == gene_b.upper())]
    if not a_to_b.empty:
        return {'trrust_regulatory_pair': True, 'trrust_direction': f'{gene_a}->{gene_b}', 'trrust_regtype': a_to_b['regtype'].iloc[0]}
    b_to_a = trrust[(trrust['tf'].str.upper() == gene_b.upper()) & (trrust['target'].str.upper() == gene_a.upper())]
    if not b_to_a.empty:
        return {'trrust_regulatory_pair': True, 'trrust_direction': f'{gene_b}->{gene_a}', 'trrust_regtype': b_to_a['regtype'].iloc[0]}
    return {'trrust_regulatory_pair': False, 'trrust_direction': None, 'trrust_regtype': None}


trrust_rows = []
for _, row in query_set.iterrows():
    result = check_trrust_regulation(row['cognate_gene'], row['top_predictor_gene'])
    result['protein'] = row['protein']
    trrust_rows.append(result)

trrust_results = pd.DataFrame(trrust_rows)
print(f"TRRUST-supported (regulatory) pairs: {trrust_results['trrust_regulatory_pair'].sum()} / {len(trrust_results)}")
trrust_results[trrust_results['trrust_regulatory_pair']]

TRRUST-supported (regulatory) pairs: 1 / 60


,trrust_regulatory_pair,trrust_direction,trrust_regtype,protein
16,True,FOXP3->IL2RA,Activation,AB_CD25


## Combined verdict
A pair counts as biologically supported if STRING, the widened complex check, or TRRUST confirms it. The evidence-type columns let you distinguish physical (complex), regulatory (TRRUST), and general-association (STRING) support, and separate hard evidence from text-mining-only STRING hits.

In [16]:
combined = string_results.merge(complex_results, on='protein').merge(trrust_results, on='protein')
combined = combined.merge(query_set[['protein', 'cognate_rank']], on='protein')

combined['biologically_supported'] = (
    combined['string_known_interaction'] | combined['shared_complex'] | combined['trrust_regulatory_pair']
)
combined['evidence_type'] = np.select(
    [combined['shared_complex'], combined['trrust_regulatory_pair'], combined['string_hard_evidence'],
     combined['string_known_interaction']],
    ['complex', 'regulatory', 'string_hard_evidence', 'string_textmining_or_other'],
    default='none',
)

print(f"Total queried: {len(combined)}")
print(f"Biologically supported (any source): {combined['biologically_supported'].sum()}")
print(f"Unexplained (no source supports): {(~combined['biologically_supported']).sum()}")
print('\nBy evidence type:')
print(combined['evidence_type'].value_counts())

combined.sort_values(['biologically_supported', 'string_score'], ascending=[False, False])

Total queried: 60
Biologically supported (any source): 28
Unexplained (no source supports): 32

By evidence type:
evidence_type
none                          32
complex                       12
string_textmining_or_other    11
string_hard_evidence           4
regulatory                     1
Name: count, dtype: int64


,protein,cognate_gene,top_predictor_gene,string_known_interaction,string_score,string_escore,string_dscore,string_tscore,string_ascore,string_nscore,...,string_hard_evidence,string_textmining_only,shared_complex,complex_names,trrust_regulatory_pair,trrust_direction,trrust_regtype,cognate_rank,biologically_supported,evidence_type
6,AB_CD8,CD8A,CD8B,True,0.999,0.691,0.9,0.892,0.819,0.0,...,True,False,True,CD8_receptor,False,None,None,2,True,complex
56,AB_KLRD1,KLRD1,KLRC1,True,0.999,0.982,0.9,0.986,0.096,0.0,...,True,False,True,CD94:NKG2A,False,None,None,3,True,complex
7,AB_CD19,CD19,CD79A,True,0.998,0.292,0.5,0.970,0.829,0.0,...,True,False,False,,False,None,None,4,True,string_hard_evidence
13,AB_CD4,CD4,CD8B,True,0.980,0.000,0.9,0.785,0.166,0.0,...,True,False,False,,False,None,None,6,True,string_hard_evidence
16,AB_CD25,IL2RA,FOXP3,True,0.970,0.288,0.5,0.916,0.142,0.0,...,True,False,False,,True,FOXP3->IL2RA,Activation,2,True,regulatory
49,AB_CD22,CD22,MS4A1,True,0.928,0.092,0.0,0.807,0.542,0.0,...,False,True,False,,False,None,None,5,True,string_textmining_or_other
35,AB_ITGB7,ITGB7,ITGB1,True,0.913,0.000,0.9,0.139,0.000,0.0,...,True,False,False,,False,None,None,45,True,string_hard_evidence
20,AB_FCGR2A,FCGR2A,FCGR2B,True,0.892,0.471,0.0,0.706,0.356,0.0,...,True,False,True,unnamed,False,None,None,359,True,complex
37,AB_SELP,SELP,PPBP,True,0.887,0.000,0.0,0.862,0.219,0.0,...,False,True,False,,False,None,None,666,True,string_textmining_or_other
32,AB_CD21,CR2,CD79A,True,0.886,0.000,0.0,0.829,0.363,0.0,...,False,True,False,,False,None,None,12,True,string_textmining_or_other


## Save results

In [ ]:
combined.to_csv(RESULTS_DIR / 'pathway_validation_results.csv', index=False)
combined[combined['biologically_supported']].to_csv(RESULTS_DIR / 'biologically_supported_pairs.csv', index=False)
combined[~combined['biologically_supported']].to_csv(RESULTS_DIR / 'unexplained_pairs.csv', index=False)

print(f'Saved to {RESULTS_DIR}')
print('  pathway_validation_results.csv   -- STRING (full evidence breakdown) + complexes + TRRUST, all queried pairs')
print('  biologically_supported_pairs.csv -- pairs confirmed by at least one source, with evidence_type')
print('  unexplained_pairs.csv            -- pairs no source supports -- candidates for literature spot-check or exclusion')